### Setting up envs and imports

In [1]:
from dotenv import load_dotenv
import os
from pageindex import PageIndexClient
import pageindex.utils as utils

load_dotenv()
PAGE_INDEX_API_KEY = os.getenv("API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
pi_client = PageIndexClient(api_key="PAGE_INDEX_API_KEY")
# print(pi_client)

if not PAGE_INDEX_API_KEY:
    print(
        "❌ Validation Failed: No 'API_KEY' entry detected within your local .env configuration."
    )
    exit(1)

clean_key = PAGE_INDEX_API_KEY.strip().replace('"', "").replace("'", "")
pi_client = PageIndexClient(api_key=clean_key)


### Testing Groq API KEY

In [2]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "What is the Capital of India?"}],
    reasoning_effort="medium",
)

print(completion.choices[0].message.content)

The capital of India is **New Delhi**.


### Loading PDF to create doc_id Of the PDF

In [3]:
# from pypdf import PdfReader
# reader = PdfReader(pdf_path)
pdf_path = "data/Policy-Document_LIC-s_New-Jeevan_Amar.pdf"
response = pi_client.submit_document(pdf_path)
doc_id = response["doc_id"]
print("Document Submitted:", doc_id)

# Health Check
status = pi_client.get_document(doc_id)["status"]
if status == "completed":
    print("Document processing completed")

Document Submitted: pi-cmrgbpceu005g01ph9qj6jz6l


### Registry — Save PDF info so doc_id is never lost

**Why this exists:**  
Right now `doc_id` only lives as a Python variable. Close the notebook and it's gone — you'd have to re-submit the PDF.

The registry is a simple JSON file (`data/pdf_registry.json`) that permanently maps each `doc_id` to the PDF's filename and a short description of what it covers.

**The description is the most important part.** Write it as a list of questions this PDF can answer, not just what the PDF is. The router (next cell) uses these descriptions to decide which PDF to search — so a vague description = bad routing.

Run `save_to_registry(...)` once per PDF you add to your system.

In [4]:
import json
import os

REGISTRY_PATH = "data/pdf_registry.json"


def load_registry() -> dict:
    """
    Reads the registry file from disk and returns it as a dict.
    If the file doesn't exist yet, returns an empty dict.
    """
    if os.path.exists(REGISTRY_PATH):
        with open(REGISTRY_PATH, "r") as f:
            return json.load(f)
    return {}


def save_to_registry(doc_id: str, pdf_path: str, description: str):
    """
    Saves a submitted PDF's doc_id + description to the registry file.
    Call this once per PDF, right after submitting it to PageIndex.

    Args:
        doc_id      : the doc_id you got back from pi_client.submit_document()
        pdf_path    : path to the PDF file (just used to extract the filename)
        description : what questions this PDF can answer (the router reads this)
    """
    registry = load_registry()  # load existing entries so we don't overwrite them

    registry[doc_id] = {
        "doc_id": doc_id,
        "filename": os.path.basename(pdf_path),
        "description": description
    }

    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)

    print(f"✅ Registered: {os.path.basename(pdf_path)} → {doc_id}")


# ── Run once per PDF ──────────────────────────────────────────────────────────
#
# NOTE: Write the description as answers to the question:
# "What kinds of questions can a user ask that this PDF would answer?"
#
# WEAK  → "LIC policy document"
# STRONG → "LIC New Jeevan Amar term insurance: death benefit amounts, premium
#            payment schedules, grace period for missed payments, policy lapse
#            and revival rules, nomination, claim filing procedures."

save_to_registry(
    doc_id=doc_id,
    pdf_path=pdf_path,
    description=(
        "LIC New Jeevan Amar term insurance policy. Covers: death benefit payout amounts, "
        "premium payment schedules, grace period for missed payments, policy lapse and "
        "revival rules, free look period, nomination and assignment procedures, "
        "surrender value, claim filing procedures, grievance redressal mechanism."
    )
)

# Verify what's saved
print("\nCurrent registry:")
print(json.dumps(load_registry(), indent=2))

✅ Registered: Policy-Document_LIC-s_New-Jeevan_Amar.pdf → pi-cmrgbpceu005g01ph9qj6jz6l

Current registry:
{
  "pi-cmrgbpceu005g01ph9qj6jz6l": {
    "doc_id": "pi-cmrgbpceu005g01ph9qj6jz6l",
    "filename": "Policy-Document_LIC-s_New-Jeevan_Amar.pdf",
    "description": "LIC New Jeevan Amar term insurance policy. Covers: death benefit payout amounts, premium payment schedules, grace period for missed payments, policy lapse and revival rules, free look period, nomination and assignment procedures, surrender value, claim filing procedures, grievance redressal mechanism."
  }
}


### Printing the Node Tree 

In [5]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    print("Simplified Tree Structure of the Document:")
    utils.print_tree(tree)
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'LIFE INSURANCE CORPORATION OF INDIA',
  'node_id': '0000',
  'summary': '# LIFE INSURANCE CORPORATION OF INDIA\n**...'},
 {'title': "LIC's New Jeevan Amar (UIN:512N350V01)",
  'node_id': '0001',
  'summary': "# LIC's New Jeevan Amar (UIN:512N350V01)..."},
 {'title': 'PART-A', 'node_id': '0002', 'summary': 'This document serves as a formal cover l...'},
 {'title': 'Free Look Period',
  'node_id': '0003',
  'summary': "This document outlines the policyholder'..."},
 {'title': 'PREAMBLE', 'node_id': '0004', 'summary': 'This document serves as the preamble for...'},
 {'title': 'SCHEDULE', 'node_id': '0005', 'summary': 'This document is a policy schedule for L...'},
 {'title': 'PART– B: DEFINITIONS',
  'node_id': '0006',
  'summary': 'This document provides a comprehensive g...'},
 {'title': 'PART– C: BENEFITS',
  'node_id': '0007',
  'prefix_summary': 'This document outlines the benefits, ser...',
  'nodes': [{'title': 'PART E', 'node_

### MultiPurpose Function for LLM Calling

In [6]:
from groq import Groq


def call_llm(prompt: str):
    client = Groq(api_key=GROQ_API_KEY)

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="medium",
        temperature=0,
        # response_format={"type": "json_object"},
    )

    return completion.choices[0].message.content


### Generating the JSON Tree

In [7]:
import json

query = "What are the conclusions in this document?"

tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = call_llm(search_prompt)
print(type(tree_search_result))
print(tree_search_result)

# result = json.loads(tree_search_result)
# print(json.dumps(result, indent=4))

<class 'str'>
{
    "thinking": "The question asks for the conclusions of the document. Conclusions are typically found in the final sections that wrap up the policy, such as governing law, statutory provisions, grievance mechanisms, and final legal clauses. In the provided tree, these are represented by the nodes toward the end of PART‑F and the subsequent statutory sections. Therefore, the relevant node IDs are those that contain the final legal and procedural statements.",
    "node_list": ["0019", "0020", "0021", "0022", "0023", "0024", "0025", "0026", "0027"]
}


In [8]:
import pageindex.utils as utils

node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

# print(type(node_map))
print(json.dumps(node_map, indent=4))

print("Reasoning Process:")
utils.print_wrapped(tree_search_result_json["thinking"])

print("\nRetrieved Nodes:")
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(
        f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}"
    )

{
    "0000": {
        "title": "LIFE INSURANCE CORPORATION OF INDIA",
        "node_id": "0000",
        "page_index": 1,
        "summary": "# LIFE INSURANCE CORPORATION OF INDIA\n**(Established by the Life Insurance Corporation Act, 1956)**\n**Registration Number: 512**\n",
        "text": "# LIFE INSURANCE CORPORATION OF INDIA\n**(Established by the Life Insurance Corporation Act, 1956)**\n**Registration Number: 512**\n"
    },
    "0001": {
        "title": "LIC's New Jeevan Amar (UIN:512N350V01)",
        "node_id": "0001",
        "page_index": 1,
        "summary": "# LIC's New Jeevan Amar (UIN:512N350V01)\n**( A Non-Linked, Non-Participating, Individual, Pure Risk Premium Life Insurance Plan)**\n",
        "text": "# LIC's New Jeevan Amar (UIN:512N350V01)\n**( A Non-Linked, Non-Participating, Individual, Pure Risk Premium Life Insurance Plan)**\n"
    },
    "0002": {
        "title": "PART-A",
        "node_id": "0002",
        "page_index": 1,
        "summary": "This docum

### Answer Generartion from retrieved Context

In [9]:
if tree_search_result is None:
	raise ValueError("tree_search_result is None")

node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print("Retrieved Context:\n")
utils.print_wrapped(relevant_content[:1000] + "...")

answer_prompt = f"""
Answer the question based on the context below.

Question: {query}
Context: {relevant_content}

Instructions:
- Imagine you are explaining this to a friend who just bought an insurance policy
- Use "you" and "your" instead of "the policyholder" or "the insured"
- Replace legal terms with plain words, e.g.:
    * "jurisdiction" → "will handle your case"
    * "repudiation" → "rejection of your claim"
    * "bona fide" → "genuine"
    * "statutory provisions" → "legal rules"
- Keep bullet points but make each one a complete, friendly sentence
- Start with a plain 1-sentence summary
- End with a "Bottom line:" that tells the reader what they should actually DO or KNOW
- Do not use bold headers for every point — only bold the most critical warnings
- Write at a reading level suitable for someone with a high school education
- Avoid all Latin phrases and insurance jargon
"""

print("\n\n\nGenerated Answer:\n")

answer = call_llm(answer_prompt)
utils.print_wrapped(answer)

Retrieved Context:

### 7. Governing Law and Jurisdiction:

The Policy shall be governed by the laws of India and the Indian Courts shall have jurisdiction to
settle any disputes arising under the Policy


### PART – G: STATUTORY PROVISIONS


### Section 45 of the Insurance Act 1938:

The provisions of Section 45 of the Insurance Act 1938, as amended from time to time, shall be
applicable. The current provisions are contained in Annexure-3 of this Policy Document.


### Grievance Redressal Mechanism:


### Of the Corporation:

The Corporation has Grievance Redressal Officers at Branch/ Divisional/ Zonal/ Central Office to
redress grievances of customers. For ensuring quick redressal of customer grievances the Corporation
has introduced Customer friendly Integrated Complaint Management System through our Customer Portal
(website) which is http://www.licindia.in, where a registered policy holder can directly register
complaint/ grievance and track its status. Customers can also contact a

---
## Multi-PDF Pipeline

Everything above works for a single hardcoded PDF. The cells below upgrade it to handle **multiple PDFs** and **different types of questions**.

### How it works (read this before running)

There are 3 question types the system handles differently:

| Type | Example | What happens |
|------|---------|-------------|
| `specific` | "What is the grace period in my LIC policy?" | Router picks the right PDF(s) → node search → answer |
| `general` | "What is term insurance?" | LLM answers directly from its own knowledge, no PDF needed |
| `ambiguous` | "Tell me about my policy" | System asks the user to clarify |

The **router** is a single LLM call that reads the short descriptions from your registry and classifies which type the question is, and which `doc_id`(s) to search.

### Router — Classify the query and pick relevant PDFs

**What this function does:**  
It sends the user's question + the registry descriptions to the LLM, and asks it to:
1. Classify the question as `specific`, `general`, or `ambiguous`  
2. Return the `doc_id`s of relevant PDFs (only if `specific`)  
3. Suggest a clarification question (only if `ambiguous`)  

Notice we are NOT sending any PDF content here — just the short descriptions.  
This makes routing fast and cheap.

In [10]:
def route_query(query: str) -> dict:
    """
    Classifies the query and returns which doc_ids are relevant.

    Returns a dict with:
        type           : 'specific' | 'general' | 'ambiguous'
        doc_ids        : list of doc_ids to search (empty if not specific)
        thinking       : the LLM's reasoning (useful for debugging)
        clarification  : question to ask the user (only when ambiguous)
    """
    registry = load_registry()

    if not registry:
        print("⚠️  Registry is empty. Submit and register at least one PDF first.")
        return {"type": "general", "doc_ids": [], "thinking": "", "clarification": ""}

    # Build a compact catalog — only what the LLM needs to make a routing decision.
    # We deliberately exclude the tree and full text; those come later.
    catalog = [
        {
            "doc_id": v["doc_id"],
            "filename": v["filename"],
            "description": v["description"]
        }
        for v in registry.values()
    ]

    routing_prompt = f"""You are a document router for an insurance Q&A assistant.
A user has asked a question. Decide which documents to search and classify the question type.

User question: {query}

Available documents:
{json.dumps(catalog, indent=2)}

Classify the question into ONE of these types:
- "specific"   : the answer is likely inside one or more of the documents above
- "general"    : general insurance knowledge question — no document needed, LLM can answer directly
- "ambiguous"  : too vague to know which document or topic to look at

Reply ONLY with this JSON, nothing else:
{{
    "thinking": "<your reasoning>",
    "type": "specific",
    "doc_ids": ["doc_id_1"],
    "clarification": ""
}}

Rules:
- If type is "general"   → doc_ids must be []
- If type is "ambiguous" → doc_ids must be [], clarification must be a useful follow-up question
- If type is "specific"  → include ALL relevant doc_ids, clarification must be ""
"""

    result = call_llm(routing_prompt)
    parsed = json.loads(result)

    # Always print this so you can see why the router made its decision
    print(f"🔍 Question type  : {parsed['type']}")
    print(f"📄 Routed to docs : {parsed['doc_ids']}")
    print(f"💭 Reasoning      : {parsed['thinking']}")

    return parsed


# ── Test the router with three different question types ───────────────────────
print("=" * 60)
print("TEST 1 — specific question")
print("=" * 60)
route_query("What happens if I miss a premium payment?")

print("\n" + "=" * 60)
print("TEST 2 — general question")
print("=" * 60)
route_query("What is term insurance?")

print("\n" + "=" * 60)
print("TEST 3 — ambiguous question")
print("=" * 60)
route_query("Tell me about my policy")

TEST 1 — specific question
🔍 Question type  : specific
📄 Routed to docs : ['pi-cmrgbpceu005g01ph9qj6jz6l']
💭 Reasoning      : The user asks about the consequences of missing a premium payment. The provided document description explicitly mentions it covers 'grace period for missed payments, policy lapse and revival rules', which directly addresses this question. Therefore the answer can be found in the specific policy document.

TEST 2 — general question
🔍 Question type  : general
📄 Routed to docs : []
💭 Reasoning      : The user asks for a definition of term insurance, which is a general insurance concept. The available document is a specific policy document that details terms and conditions of a particular product, not a general definition. The answer can be provided from general knowledge without needing to search the document.

TEST 3 — ambiguous question
🔍 Question type  : ambiguous
📄 Routed to docs : []
💭 Reasoning      : The user asks a very vague question 'Tell me about my poli

{'thinking': "The user asks a very vague question 'Tell me about my policy' without specifying which policy. We have only one policy document available, but we cannot assume it is the user's policy. Therefore the request is ambiguous and we need clarification.",
 'type': 'ambiguous',
 'doc_ids': [],
 'clarification': 'Could you please specify which policy you would like information about (e.g., policy name, number, or type)?'}

### Node Search — wrapped as a reusable function

This is the same logic from the cells above (the JSON tree search + context retrieval),  
just wrapped into a function so the main pipeline can call it for each routed PDF.

One new thing: each retrieved chunk is now **tagged with its source filename and page number**.  
This lets the answer generation step tell the user *where* the info came from.

In [11]:
def search_nodes(doc_id: str, query: str) -> list[str]:
    """
    Searches the node tree of a single PDF for content relevant to the query.
    Returns a list of text chunks, each tagged with [Source: filename, Page N].

    This is your existing pipeline (tree fetch → LLM node search → retrieve text),
    just wrapped in a function so it can be called per-document in a loop.
    """
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"⚠️  Doc {doc_id} not ready yet — skipping.")
        return []

    # Step 1: fetch the tree for this specific doc
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    # Step 2: ask LLM which nodes are relevant (same prompt as before)
    node_search_prompt = f"""You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

    node_result = json.loads(call_llm(node_search_prompt))
    node_map = utils.create_node_mapping(tree)

    # Step 3: retrieve text from the matched nodes
    # We look up the filename from the registry to tag the source
    registry = load_registry()
    filename = registry[doc_id]["filename"]

    chunks = []
    for node_id in node_result["node_list"]:
        if node_id not in node_map:
            continue
        node = node_map[node_id]
        # Tag each chunk so the LLM knows which document and page it came from
        tagged_chunk = f"[Source: {filename}, Page {node['page_index']}]\n{node['text']}"
        chunks.append(tagged_chunk)

    print(f"  📑 {filename}: found {len(chunks)} relevant node(s)")
    return chunks

### ask() — The main pipeline that ties everything together

This is the single function you call for any user query.  
Internally it:
1. Calls `route_query()` to classify the question  
2. Branches based on the type:  
   - `ambiguous` → prints the clarification question, stops  
   - `general`   → answers directly from LLM knowledge, no PDF  
   - `specific`  → loops over routed `doc_ids`, calls `search_nodes()` for each, combines context, generates answer  

In [12]:
def ask(query: str):
    """
    Main entry point. Pass any user question — the system routes and answers it.
    """
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print("=" * 60)

    # ── Step 1: Route ─────────────────────────────────────────────────────────
    routing = route_query(query)
    q_type = routing["type"]

    # ── Branch: Ambiguous ─────────────────────────────────────────────────────
    # The question is too vague — ask the user to clarify before doing anything
    if q_type == "ambiguous":
        print(f"\n🤔 Could you clarify: {routing['clarification']}")
        return

    # ── Branch: General ───────────────────────────────────────────────────────
    # General insurance knowledge — the LLM already knows this, no PDF needed
    elif q_type == "general":
        print("\n💡 General question — answering from LLM knowledge directly.\n")
        general_prompt = f"""Answer this insurance question clearly and simply.
Use plain language suitable for someone who just bought their first insurance policy.
Avoid jargon. Start with a one-sentence summary.

Question: {query}
"""
        answer = call_llm(general_prompt)
        utils.print_wrapped(answer)

    # ── Branch: Specific ──────────────────────────────────────────────────────
    # The question is tied to specific PDFs — run the full RAG pipeline
    elif q_type == "specific":
        print(f"\n🔎 Searching {len(routing['doc_ids'])} document(s)...")

        all_chunks = []

        # Loop over every routed doc_id and collect relevant chunks from each
        # This is how multi-PDF support works — same search_nodes() function,
        # just called once per relevant document
        for doc_id in routing["doc_ids"]:
            chunks = search_nodes(doc_id, query)
            all_chunks.extend(chunks)

        if not all_chunks:
            print("❌ No relevant content found in the matched documents.")
            return

        # Join all chunks with a separator so the LLM can distinguish sources
        context = "\n\n---\n\n".join(all_chunks)

        answer_prompt = f"""Answer the question based on the context below.
If the context comes from multiple documents, mention which document each point is from.

Question: {query}
Context: {context}

Instructions:
- Imagine you are explaining this to a friend who just bought an insurance policy
- Use "you" and "your" instead of "the policyholder" or "the insured"
- Replace legal terms with plain words
- Start with a plain 1-sentence summary
- End with a "Bottom line:" that tells the reader what they should actually DO or KNOW
- Only bold the most critical warnings
- Write at a reading level suitable for someone with a high school education
"""

        print("\n📝 Generated Answer:\n")
        answer = call_llm(answer_prompt)
        utils.print_wrapped(answer)


# ── Try it ────────────────────────────────────────────────────────────────────
ask("What happens if I miss a premium payment?")


Query: What happens if I miss a premium payment?
🔍 Question type  : specific
📄 Routed to docs : ['pi-cmrgbpceu005g01ph9qj6jz6l']
💭 Reasoning      : The user asks about the consequences of missing a premium payment. The provided document description explicitly mentions it covers 'grace period for missed payments, policy lapse and revival rules', which directly addresses this question. Therefore the answer can be found in the specific policy document.

🔎 Searching 1 document(s)...
  📑 Policy-Document_LIC-s_New-Jeevan_Amar.pdf: found 2 relevant node(s)

📝 Generated Answer:

**Missing a premium means you have a short window to fix it, or the policy will lapse and you lose
the cover.**

- **Grace period (Page 6):** After a premium is due, you get 30 days to pay it. If you pay within
those 30 days, the policy stays active.
- **If you die during the grace period:** The policy is still considered valid, but the unpaid
premium (and any other premiums that would have become due before the next 

In [13]:
# Try more queries to test all three branches

ask("What is term insurance?")           # should hit: general
ask("Tell me about my policy")           # should hit: ambiguous
ask("How do I file a claim after death?") # should hit: specific


Query: What is term insurance?
🔍 Question type  : general
📄 Routed to docs : []
💭 Reasoning      : The user asks for a definition of term insurance, which is a general insurance concept. The available document is a specific policy document that details terms and conditions of a particular product, not a general definition. The answer can be provided from general knowledge without needing to search the document.

💡 General question — answering from LLM knowledge directly.

**Term insurance is a life‑insurance plan that pays a benefit only if you die during a set number of
years.**

It works like this:

- **Fixed time period:** You choose how long the coverage lasts—often 10, 20, or 30 years.
- **Pay‑as‑you‑go:** You pay a regular premium (monthly or yearly) for the whole term.
- **Benefit only if you pass away during the term:** If you die while the policy is active, the
insurer gives a lump‑sum payment to your chosen beneficiaries.
- **No payout after the term ends:** If you outlive t